In [46]:
import json
import sagemaker

from sagemaker.inputs import TrainingInput

from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel

from sagemaker.workflow.parameters import ParameterInteger, ParameterString, ParameterBoolean
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.lambda_step import LambdaStep, Lambda

#from sagemaker.workflow.step_collections import RegisterModel # Para registro y auditoría del modelo y luego despliegue fuera de un pipeline
from sagemaker.workflow.model_step import ModelStep # Para registro y auditoría del modelo como step de pipeline y luego despliegue
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.pipeline_context import PipelineSession

In [47]:
env_prefix = "develop"
default_bucket = f"pipeline-test-ml-sklearn-randomforest-artifacts"

sagemaker_session = sagemaker.Session(
    default_bucket=default_bucket,
    default_bucket_prefix=env_prefix
)
pipeline_session = PipelineSession(
    default_bucket=default_bucket,
    default_bucket_prefix=env_prefix
)

default_bucket = f"pipeline-test-ml-sklearn-randomforest-artifacts/{env_prefix}"

sm_client = sagemaker_session.sagemaker_client
region = sagemaker_session.boto_region_name
role = "arn:aws:iam::007863746889:role/sagemakerS3"

account_id = sagemaker_session.account_id()

In [48]:
prefix_input_data = "data/raw/"
base_job_prefix = "randomForest-pipeline"

# Parameter aws instances
processing_instance_count = 1
training_instance_count = 1

processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value="ml.t3.large")
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.large")

# Parameter Model / Paths
input_data = ParameterString(name="InputRawData", default_value=f"s3://{default_bucket}/{prefix_input_data}")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")
run_hiperparameter_tuner = ParameterBoolean(name="RunHiperTuner", default_value=True)

# Cache Pipeline steps to reduce execution time on subsequent executions
cache_config = CacheConfig(enable_caching=True, expire_after="10d")

# Scripts of the pipeline steps
pipeline_scripts_s3 = f"s3://{default_bucket}/code/scripts"

# Preprocessing

In [ ]:
# Process the training data step using a python script.
# Split the training data set into train, test, and validation datasets

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name=f"{base_job_prefix}/sklearn-LoanDefault-preprocess",
    sagemaker_session=pipeline_session,
    role=role,
    
)

processor_args = {
    "outputs": [
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{default_bucket}/data/train/"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation",
            destination=f"s3://{default_bucket}/data/validation/"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{default_bucket}/data/test/"
        ),
        ProcessingOutput(
            output_name="tranformer",
            source="/opt/ml/processing/transformer",
            destination=f"s3://{default_bucket}/model/transformer/"
        )
    ],
    "inputs": [
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input/input_data",
            input_name="input_data",
            s3_input_mode="File"
        )
    ],
    "code": f"code/preprocess.py",
    "arguments": ["--input-data", "/opt/ml/processing/input/input_data"]
}
processor_args = sklearn_processor.run(**processor_args)

step_process = ProcessingStep(
    name="PreprocessLoanDefaultData",
    step_args=processor_args,
    cache_config=cache_config
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


# Training

In [50]:
min_samples_split = ParameterInteger(name="MinSamplesSplit", default_value=10)
class_weight = ParameterString(name="ClassWeight", default_value='{"0":"1","1":"25"}')
max_depth = ParameterInteger(name="MaxDepth", default_value=20)
n_estimator = ParameterInteger(name="NEstimators", default_value=100)

sklearn_train_estimator = SKLearn(
    entry_point="code/train.py",
    framework_version="1.2-1",
    instance_type=training_instance_type,
    instance_count=training_instance_count,
    output_path=f"s3://{default_bucket}/model_artifacts",
    script_mode=True,
    role=role,
    py_version="py3",
    base_job_name=f"{base_job_prefix}/sklearn-LoanDefault-training",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": n_estimator,
        "max-depth": max_depth,
        "class-weight": class_weight,
        "min-samples-split": min_samples_split
    }
)

train_args = {
    "inputs": {
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type= "text/csv"
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
}
train_args = sklearn_train_estimator.fit(**train_args)

step_train = TrainingStep(
    name="TrainSklearnLoanDefaultModel",
    step_args=train_args,
    cache_config=cache_config
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


# Evaluation

In [51]:
evaluation_processor = ScriptProcessor(
    role=role,
    image_uri=sklearn_train_estimator.image_uri,
    instance_count=training_instance_count,
    instance_type=processing_instance_type,
    base_job_name=f"{base_job_prefix}/evaluationLoanDefaultModel",
    sagemaker_session=pipeline_session,
    command=["python3"]
)

evaluation_args = {
    "inputs":[
        ProcessingInput(
            input_name="Model",
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            input_name="train_data",
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/data/test"
        )
    ],
    "outputs":[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=f"s3://{default_bucket}/code/evaluation_report"
        )
    ],
    "code":"code/evaluation.py",
}
evaluation_report = PropertyFile(
    name="LoanDefaultEvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

evaluation_args = evaluation_processor.run(**evaluation_args)

step_evaluation = ProcessingStep(
    name="EvaluateSKlearnLoanDefaultModel",
    step_args=evaluation_args,
    cache_config=cache_config,
    property_files=[evaluation_report]
)

# Model and conditional step

In [52]:
sklearn_model = SKLearnModel(
    entry_point="code/inference.py",
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    image_uri=sklearn_train_estimator.image_uri,
    sagemaker_session=pipeline_session
)

sklearn_model_args = dict(
    content_types=["text/csv"],
    response_types=["text/csv"],
    model_package_group_name="LoanDefaultSKLeanrModelGroup",
    approval_status="Approved"
)

sklearn_model_step_auto = ModelStep(
    name="LoanDefaultApprovedModel",
    step_args=sklearn_model.register(**sklearn_model_args)
)

sklearn_model_args["approval_status"] = "Rejected"
sklearn_model_step_rejected = ModelStep(
    name="LoanDefaultRejectedModel",
    step_args=sklearn_model.register(**sklearn_model_args)
)

In [53]:
f1_score = JsonGet(
    step_name=step_evaluation.name,
    property_file=evaluation_report,
    json_path="classification_metrics.test.f1"
)

# Automatic lambda deployment (severless)
lambda_deploy = LambdaStep(
    name="DeployModelWithLambda",
    lambda_func=Lambda(
        function_arn="arn:aws:lambda:us-east-1:007863746889:function:SeverlessDeploySagemakerPIpeline",
        session=pipeline_session,
    ),
    inputs={
        "model_package_arn": sklearn_model_step_auto.properties.ModelPackageArn
    }
)

condition_step = ConditionStep(
    name="CheckF1Score",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=f1_score,
            right=0.5
        )
    ],
    if_steps=[sklearn_model_step_auto, lambda_deploy],
    else_steps=[sklearn_model_step_rejected]
)

# Pipeline Definition

In [54]:
params = [
    input_data,
    processing_instance_type,
    training_instance_type,
    min_samples_split,
    class_weight,
    max_depth,
    n_estimator
]

pipeline_instance = Pipeline(
    name=f"PipelineSkLernLoanDefault",
    parameters=params,
    steps=[step_process, step_train, step_evaluation, condition_step],
    sagemaker_session=pipeline_session
)

pipeline_definition = json.loads(pipeline_instance.definition())
pipeline_definition

{'Version': '2020-12-01',
 'Metadata': {},
 'Parameters': [{'Name': 'InputRawData',
   'Type': 'String',
   'DefaultValue': 's3://pipeline-test-ml-sklearn-randomforest-artifacts/develop/data/raw/'},
  {'Name': 'ProcessingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.t3.large'},
  {'Name': 'TrainingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.large'},
  {'Name': 'MinSamplesSplit', 'Type': 'Integer', 'DefaultValue': 10},
  {'Name': 'ClassWeight',
   'Type': 'String',
   'DefaultValue': '{"0":"1","1":"25"}'},
  {'Name': 'MaxDepth', 'Type': 'Integer', 'DefaultValue': 20},
  {'Name': 'NEstimators', 'Type': 'Integer', 'DefaultValue': 100}],
 'PipelineExperimentConfig': {'ExperimentName': {'Get': 'Execution.PipelineName'},
  'TrialName': {'Get': 'Execution.PipelineExecutionId'}},
 'Steps': [{'Name': 'PreprocessLoanDefaultData',
   'Type': 'Processing',
   'Arguments': {'ProcessingResources': {'ClusterConfig': {'InstanceType': {'Get': 'Parameters.ProcessingInstanc

In [55]:
pipeline_instance.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:007863746889:pipeline/PipelineSkLernLoanDefault',
 'ResponseMetadata': {'RequestId': '854589f6-21a6-4c48-8981-2968864ecbe7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '854589f6-21a6-4c48-8981-2968864ecbe7',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '115',
   'date': 'Mon, 28 Jul 2025 19:52:11 GMT'},
  'RetryAttempts': 0}}

In [56]:
execution = pipeline_instance.start()
execution.wait()